<a href="https://colab.research.google.com/github/raghad-cs/Esnad/blob/feature%2Falarb-pipeline/alarb_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets sentence-transformers faiss-cpu pandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.2 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

dataset = load_dataset("THIQAH-RD/ALARB")

print(dataset)
print(dataset["train"].column_names)

README.md:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 19.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.15MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12012 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1329 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['case_facts', 'court_reasoning', 'applicable_laws', 'verdict'],
        num_rows: 12012
    })
    test: Dataset({
        features: ['case_facts', 'court_reasoning', 'applicable_laws', 'verdict'],
        num_rows: 1329
    })
})
['case_facts', 'court_reasoning', 'applicable_laws', 'verdict']


In [4]:
sample = dataset["train"].select(range(50))
df = sample.to_pandas()

print("عدد القضايا:", len(df))
print("الأعمدة:", df.columns.tolist())

display(df.head(2))

عدد القضايا: 50
الأعمدة: ['case_facts', 'court_reasoning', 'applicable_laws', 'verdict']


,case_facts,court_reasoning,applicable_laws,verdict
0,[1- بتاريخ 1443/09/06 اتفق أطراف الدعوى على أن...,[1- بناءً على الدعوى والإجابة، طلب وكيل المدعي...,[نظام المحاكم التجارية:22: ١.تحيل الإدارة المخ...,إثبات الصلح بين الطرفين وإلزام المدعى عليها بس...
1,[1. سبق أن تم رفع دعوى من مصنع تكنولوجيا الحدي...,[1. اعتبرت الدائرة أن اختصاص النظر في النزاع ي...,[نظام المحاكم التجارية:16: تختص المحكمة بالنظر...,"إلزام المدعى عليها بدفع مبلغ 72,000 ريال للمدع..."


In [5]:
import re
import numpy as np
import pandas as pd

def clean_text(value):
    # إذا كانت البيانات قائمة، نجمع عناصرها في نص واحد
    if isinstance(value, (list, tuple, np.ndarray)):
        value = " ".join(str(item) for item in value if item is not None)

    elif value is None:
        return ""

    else:
        value = str(value)

    # إزالة الرموز المخفية والمسافات الزائدة
    value = re.sub(r"[\u200b-\u200f\u202a-\u202e\u2060\ufeff]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


columns_to_clean = [
    "case_facts",
    "court_reasoning",
    "applicable_laws",
    "verdict"
]

# تنظيف الأعمدة الأربعة
for column in columns_to_clean:
    df[column] = df[column].apply(clean_text)


# إعطاء رقم داخلي لكل قضية
df["case_id"] = range(1, len(df) + 1)


# إنشاء النص الذي سيدخل إلى BGE-M3
df["search_text"] = (
    "وقائع القضية: " + df["case_facts"] +
    "\nتسبيب المحكمة: " + df["court_reasoning"] +
    "\nالأنظمة والمواد المطبقة: " + df["applicable_laws"] +
    "\nالحكم النهائي: " + df["verdict"]
)


print("تم تجهيز", len(df), "قضية")
print("\nأول نص جاهز للـEmbedding:\n")
print(df.loc[0, "search_text"][:1500])

تم تجهيز 50 قضية

أول نص جاهز للـEmbedding:

وقائع القضية: 1- بتاريخ 1443/09/06 اتفق أطراف الدعوى على أن تورد المدعية للمدعى عليها عمالة بثمن إجمالي قدره 69,129.60 ريال. 2- بدأ التعامل بين الطرفين بتاريخ 1442/10/13 ولم يسدد من الثمن شيء، مع استلام المدعى عليها كامل المبيع وكانت مدة العقد ثلاثة أشهر. 3- نشأ حق المدعية في استلام المبلغ بتاريخ 1444/02/02 وطالبت المدعى عليها بسداد المبلغ وتعويض قدره 6,900 ريال عن أضرار التقاضي. 4- قدمت المدعية مستندات لدعم طلبها، منها كشف حساب بتاريخ يوليو 2022 بمبلغ 69,129.60 ريال وأمر شراء بتاريخ 07/04/2022 موقع من المدعى عليها. 5- عقدت الدائرة جلسة في 1444/11/10 حضر فيها وكلاء الطرفين، طلب وكيل المدعى عليها مهلة للجواب وبحسب المادة 22 من نظام المحاكم التجارية ألزِم بتقديم الجواب فوراً. 6- أقر وكيل المدعى عليها بالمبلغ كاملاً وطلب الصلح، كما طلب وكيل المدعية مهلة لعرض الصلح على موكلته وتم تأجيل القضية. 7- في جلسة لاحقة بتاريخ 24/11/1444 عرض وكيل المدعى عليها سداد المبلغ على ستة أشهر تبدأ من أغسطس 2023، ورفضت المدعية العرض ووافقت على الجدولة على أربعة أشه

In [6]:
# معلومات الجدول وأنواع الأعمدة
df.info()

print("\nالحقول الفارغة:")
print(df[[
    "case_facts",
    "court_reasoning",
    "applicable_laws",
    "verdict"
]].eq("").sum())

print("\nعدد القضايا المكررة:")
print(df.duplicated(subset=["search_text"]).sum())


# حساب عدد كلمات كل قضية
df["word_count"] = df["search_text"].str.split().str.len()

print("\nإحصائيات عدد الكلمات:")
display(df["word_count"].describe().to_frame())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   case_facts       50 non-null     object
 1   court_reasoning  50 non-null     object
 2   applicable_laws  50 non-null     object
 3   verdict          50 non-null     object
 4   case_id          50 non-null     int64 
 5   search_text      50 non-null     object
dtypes: int64(1), object(5)
memory usage: 2.5+ KB

الحقول الفارغة:
case_facts         0
court_reasoning    0
applicable_laws    3
verdict            0
dtype: int64

عدد القضايا المكررة:
0

إحصائيات عدد الكلمات:


,word_count
count,50.000000
mean,495.640000
std,167.389776
min,223.000000
25%,400.000000
50%,478.500000
75%,578.500000
max,979.000000


In [7]:
metadata_path = "/content/judgments_metadata_50.parquet"

df.to_parquet(metadata_path, index=False)

print("تم حفظ البيانات في:")
print(metadata_path)
print("عدد الصفوف المحفوظة:", len(df))

تم حفظ البيانات في:
/content/judgments_metadata_50.parquet
عدد الصفوف المحفوظة: 50


**BGE-M3**

In [8]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

print("سيتم تشغيل المودل على:", device)

model = SentenceTransformer(
    "BAAI/bge-m3",
    device=device
)

print("تم تحميل BGE-M3 بنجاح")
print("حجم الـEmbedding:", model.get_sentence_embedding_dimension())

سيتم تشغيل المودل على: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

تم تحميل BGE-M3 بنجاح
حجم الـEmbedding: 1024


/tmp/ipykernel_826/1652542946.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("حجم الـEmbedding:", model.get_sentence_embedding_dimension())


In [9]:
import numpy as np

# Convert the search text column into a list
texts = df["search_text"].tolist()

# Generate normalized embeddings for all cases
embeddings = model.encode(
    texts,
    batch_size=2,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# FAISS requires float32 vectors
embeddings = embeddings.astype("float32")

print("Embeddings generated successfully.")
print("Embeddings shape:", embeddings.shape)
print("Data type:", embeddings.dtype)
print("First vector norm:", np.linalg.norm(embeddings[0]))

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Embeddings generated successfully.
Embeddings shape: (50, 1024)
Data type: float32
First vector norm: 1.0


**FAISS**

In [10]:
import faiss

# Get the embedding dimension
embedding_dimension = embeddings.shape[1]

# Create an exact cosine-similarity index
index = faiss.IndexFlatIP(embedding_dimension)

# Add all case embeddings to the index
index.add(embeddings)

print("FAISS index created successfully.")
print("Embedding dimension:", index.d)
print("Number of indexed cases:", index.ntotal)

FAISS index created successfully.
Embedding dimension: 1024
Number of indexed cases: 50


In [11]:
index_path = "/content/judgments_50.index"

faiss.write_index(index, index_path)

print("FAISS index saved to:", index_path)

FAISS index saved to: /content/judgments_50.index


Test it

In [12]:
# Natural-language user query
user_query = """
شركة جابت عمال لشركة ثانية، والطرف الثاني استلم الخدمة
لكنه ما دفع المبلغ، وبعدها اتفقوا على تسوية الدين.
"""

# Convert the query into an embedding using the same model
query_embedding = model.encode(
    [user_query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Search for the five most similar cases
top_k = 5
similarity_scores, case_positions = index.search(
    query_embedding,
    top_k
)

print("Search completed successfully.")
print("Returned positions:", case_positions[0])
print("Similarity scores:", similarity_scores[0])

Search completed successfully.
Returned positions: [ 0 34 33  3 36]
Similarity scores: [0.5752697  0.53831184 0.5346795  0.5262228  0.52569467]


In [13]:
results = df.iloc[case_positions[0]].copy()

results["similarity_score"] = similarity_scores[0]
results["facts_preview"] = (
    results["case_facts"].str[:350] + "..."
)

display(
    results[
        [
            "case_id",
            "similarity_score",
            "facts_preview",
            "verdict"
        ]
    ]
)

,case_id,similarity_score,facts_preview,verdict
0,1,0.575270,1- بتاريخ 1443/09/06 اتفق أطراف الدعوى على أن ...,إثبات الصلح بين الطرفين وإلزام المدعى عليها بس...
34,35,0.538312,1. تعاقد المدعي مع المدعى عليه لتنفيذ توريد أي...,"إلزام المدعى عليها بسداد 245,621.35 ريال للمدع..."
33,34,0.534679,اتفق المدعية والمدعى عليه بتاريخ 3/4/1438هـ عل...,الحكم بإثبات الصلح بين الطرفين وإلزامهما بمضمو...
3,4,0.526223,1. تقدم المدعي بلائحة دعوى إلى المحكمة التجاري...,إثبات الصلح الملزم للطرفين بسداد المدعى عليها ...
36,37,0.525695,1. تقدم المدعي بلائحة دعوى إلى المحكمة التجاري...,حكمت المحكمة بعدم قبول الدعوى لإقامتها على غير...
